# Plot Heatmap of DE genes per condition

In [ ]:
%autosave 0

In [ ]:
import os
import sys
from pathlib import Path

# Set the base project directory
base_proj_dir = Path("/projects/site/pred/ihb-g-deco/USERS/adaml9/intestine_fate/notebooks/tf_ko_panel_analysis")

# Append necessary paths for module imports
sys.path.append("/projects/site/pred/ihb-g-deco/USERS/adaml9/bioinfo-blueprint/src/python")
sys.path.append(str(base_proj_dir))

import anndata as ad
import numpy as np
import pandas as pd
import scanpy as sc
import seaborn as sns
import delnx as dx
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors as mcolors
import plotting.settings
from dynaconf import Dynaconf
from tqdm import tqdm
from functools import reduce
import numpy as np
import pandas as pd
import marsilea as ma 
import marsilea.plotter as mp 
import scipy.cluster.hierarchy as sch
from collections import OrderedDict

# Load settings
settings = Dynaconf(
    settings_files=[base_proj_dir / 'settings.toml', base_proj_dir / '.secrets.toml'],
)

sc.settings.figdir = settings.ANALYSIS.figures_dir

In [ ]:
data_dir = Path(settings.IO.combined_data) / "all-sample" / "DGE_filtered"
anndata_dir = Path(settings.IO.anndata)

In [ ]:
# Subset to cell types
include_cell_types = ["Stem_Cells", "TA_Cells", "Cycling_TA_Cells", "Enterocytes", "Goblet_Cells", "EEC_Progenitors", "EC_Cells", "X_Cells", "I_N_Cells", "K_Cells"]

In [ ]:
# Flexible config
sample_key   = "sample"
groupby_key = "annotation"
condition_key = "condition"
collapse_keys = [groupby_key, condition_key]      

## Load pseudobulk adata

In [ ]:
adata_pb = sc.read(anndata_dir / "tf_ko_panel_contrastiveVI_annotated_pb.h5ad")

# Format groupby key
adata_pb.obs[groupby_key] = adata_pb.obs[groupby_key].astype(str).str.replace(" ", "_").str.replace("/", "_")

# log1p transform the CPM values
sc.pp.log1p(adata_pb, layer="cpm")

# Filter for EEC subtypes 
adata_pb = adata_pb[adata_pb.obs[groupby_key].isin(include_cell_types)]

# Filter for sufficient cells
adata_pb = adata_pb[adata_pb.obs["psbulk_cells"] >= 20]

# Merge annotation with condition column
adata_pb.obs["_group"] = adata_pb.obs[groupby_key].astype(str) + "__" + adata_pb.obs[condition_key].astype(str)

# Reorder categories
adata_pb.obs[groupby_key] = adata_pb.obs[groupby_key].astype("category").cat.reorder_categories(include_cell_types)

## Load DE results

In [ ]:
# Load DE results
de_results = pd.read_csv('/projects/site/pred/ihb-intestine-evo/lukas_area/eec_fate_project/tf_ko_screen/panel/tables/annotation_de_results/tf_ko_panel_contrastiveVI_de_results_annotations.tsv', sep="\t")

# Rename columns
de_results = de_results.rename(columns={"annotation": "group", "gene_name": "feature"})

# Format group names
de_results["group"] = de_results["group"].str.replace(" ", "_").str.replace("/", "_")

# Filter for included cell types
de_results = de_results[de_results["group"].isin(include_cell_types)]

coef_col = "coef"
pval_col = "padj"
pval_thresh = 0.05
coef_thresh = 0.5

sig_mask = (de_results[pval_col] < pval_thresh)
up_mask = sig_mask & (de_results[coef_col] > coef_thresh)
down_mask = sig_mask & (de_results[coef_col] < -coef_thresh)

de_results["significant"] = "NS"
de_results.loc[up_mask, "significant"] = "Up"
de_results.loc[down_mask, "significant"] = "Down"

In [ ]:
de_results["cell_type_cond"] = de_results["group"] + "__" + de_results["condition"]

### Parse DE results and filter for upregulated genes

In [ ]:
def group_by_max(expr: pd.DataFrame) -> list[str]:
    """Order genes by group (column) of max expression."""
    max_group = expr.idxmax(axis=1)
    ordered = []
    for group in expr.columns:
        ordered.extend(max_group[max_group == group].index.tolist())
    return ordered
    
for tf_query in de_results["test_condition"].unique():
    conditions_query = ["Control", tf_query]
    adata_pb_condition = adata_pb[adata_pb.obs[condition_key].isin(conditions_query)]

    # Subset to relevant condition pairs           
    include_cond = list(adata_pb_condition.obs["_group"].unique())

    # Collect DE sets
    de_sets = (
        de_results.query("significant == 'Up' & cell_type_cond in @include_cond")
            .groupby(["group", "test_condition"])["feature"]
            .apply(set)
            .to_dict()
    )

    if len(de_sets) == 0:
        print(f"No DE genes for TF KO: {tf_query}, skipping heatmap.")
        continue

    de_sets = {
        "_".join((cell, cond)): list(genes) 
        for (cell, cond), genes in de_sets.items()
    }

    # Flatten in order of de_sets keys, keep first appearance
    flattened_genes = []
    for genes in de_sets.values():
        for g in genes:
            if g not in flattened_genes:   # preserve first appearance
                flattened_genes.append(g)

    # Intersect with adata genes, preserving order
    adata_genes = set(adata_pb_condition.var_names)
    all_de_genes = [g for g in flattened_genes if g in adata_genes]

    # Subset AnnData to relevant groups
    expr = sc.get.obs_df(
        adata_pb_condition,
        list(all_de_genes) + collapse_keys,
    )

    # Mean expression per groupby combination
    mean_expr = expr.groupby(collapse_keys)[list(all_de_genes)].mean().reset_index()

    # Melt into tidy form
    mean_expr = pd.melt(
        mean_expr,
        id_vars=collapse_keys,
        value_vars=list(all_de_genes),
        var_name="gene",
        value_name="expression"
    )

    # Pivot to wide form: genes × (annotation, condition)
    heatmap_df = mean_expr.pivot_table(
        index="gene",
        columns=collapse_keys,
        values="expression"
    )
    # Min Max scale per gene
    heatmap_df = (heatmap_df - heatmap_df.min(axis=1).values[:, None]) / (heatmap_df.max(axis=1) - heatmap_df.min(axis=1)).values[:, None]
    # Flatten MultiIndex columns
    heatmap_df.columns = ["__".join(map(str, col)) for col in heatmap_df.columns]
    heatmap_df = heatmap_df.loc[all_de_genes]

    anno_df = heatmap_df.columns.str.split("__", expand=True)
    anno_df = anno_df.to_frame().rename(columns={0: groupby_key, 1: condition_key})
    anno_df["group"] = heatmap_df.columns

    # Get all annotations with at least two groups
    annotation_counts = anno_df.groupby("annotation").size()
    include_cell_types = annotation_counts[annotation_counts >= 2].index.tolist()
    anno_df = anno_df[anno_df["annotation"].isin(include_cell_types)]

    heatmap_df = heatmap_df[anno_df["group"].tolist()]
    
    # Order genes by group of max expression
    order_genes = group_by_max(heatmap_df)
    heatmap_df = heatmap_df.loc[order_genes]

    base_ct_colors = {}
    for section, values in settings["ct_palette"].items():
        base_ct_colors.update(values)
        
    base_ct_colors = {k.replace(" ", "_").replace("/", "_"): v for k, v in base_ct_colors.items()}
    base_ct_colors = {k: v for k, v in base_ct_colors.items() if k in anno_df[groupby_key].unique()}

    h = ma.Heatmap(
        heatmap_df,
        cmap="RdPu",
        vmin=0, vmax=1,
        width=3, height=5,
        rasterized=True,
    )

    h.add_bottom(mp.Labels(heatmap_df.columns, align="center"), pad=0.1)

    h.add_top(mp.Colors(anno_df["condition"], palette={"Control": "lightgrey", tf_query: "#555555"}),
            size=.1, pad=.1)
    h.add_top(mp.Colors(anno_df["annotation"], palette=base_ct_colors),
            size=.1, pad=.01)
    h.add_legends()
    h.render()

    outdir = Path(sc.settings.figdir) / "DE_11122025" / "per_condition_heatmaps"
    outdir.mkdir(parents=True, exist_ok=True)

    plt.savefig(outdir / f"tf_ko_panel_DE_heatmap_per_condition_{tf_query}_up.pdf", bbox_inches="tight")

In [ ]:
def group_by_max(expr: pd.DataFrame) -> list[str]:
    """Order genes by group (column) of max expression."""
    max_group = expr.idxmax(axis=1)
    ordered = []
    for group in expr.columns:
        ordered.extend(max_group[max_group == group].index.tolist())
    return ordered
    
for tf_query in de_results["test_condition"].unique():
    conditions_query = ["Control", tf_query]
    adata_pb_condition = adata_pb[adata_pb.obs[condition_key].isin(conditions_query)]

    # Subset to relevant condition pairs           
    include_cond = list(adata_pb_condition.obs["_group"].unique())

    # Collect DE sets
    de_sets = (
        de_results.query("significant == 'Down' & cell_type_cond in @include_cond")
            .groupby(["group", "test_condition"])["feature"]
            .apply(set)
            .to_dict()
    )

    if len(de_sets) == 0:
        print(f"No DE genes for TF KO: {tf_query}, skipping heatmap.")
        continue

    de_sets = {
        "_".join((cell, cond)): list(genes) 
        for (cell, cond), genes in de_sets.items()
    }

    # Flatten in order of de_sets keys, keep first appearance
    flattened_genes = []
    for genes in de_sets.values():
        for g in genes:
            if g not in flattened_genes:   # preserve first appearance
                flattened_genes.append(g)

    # Intersect with adata genes, preserving order
    adata_genes = set(adata_pb_condition.var_names)
    all_de_genes = [g for g in flattened_genes if g in adata_genes]

    # Subset AnnData to relevant groups
    expr = sc.get.obs_df(
        adata_pb_condition,
        list(all_de_genes) + collapse_keys,
    )

    # Mean expression per groupby combination
    mean_expr = expr.groupby(collapse_keys)[list(all_de_genes)].mean().reset_index()

    # Melt into tidy form
    mean_expr = pd.melt(
        mean_expr,
        id_vars=collapse_keys,
        value_vars=list(all_de_genes),
        var_name="gene",
        value_name="expression"
    )

    # Pivot to wide form: genes × (annotation, condition)
    heatmap_df = mean_expr.pivot_table(
        index="gene",
        columns=collapse_keys,
        values="expression"
    )
    # Min Max scale per gene
    heatmap_df = (heatmap_df - heatmap_df.min(axis=1).values[:, None]) / (heatmap_df.max(axis=1) - heatmap_df.min(axis=1)).values[:, None]
    # Flatten MultiIndex columns
    heatmap_df.columns = ["__".join(map(str, col)) for col in heatmap_df.columns]
    heatmap_df = heatmap_df.loc[all_de_genes]

    anno_df = heatmap_df.columns.str.split("__", expand=True)
    anno_df = anno_df.to_frame().rename(columns={0: groupby_key, 1: condition_key})
    anno_df["group"] = heatmap_df.columns

    # Get all annotations with at least two groups
    annotation_counts = anno_df.groupby("annotation").size()
    include_cell_types = annotation_counts[annotation_counts >= 2].index.tolist()
    anno_df = anno_df[anno_df["annotation"].isin(include_cell_types)]

    heatmap_df = heatmap_df[anno_df["group"].tolist()]
    
    # Order genes by group of max expression
    order_genes = group_by_max(heatmap_df)
    heatmap_df = heatmap_df.loc[order_genes]

    base_ct_colors = {}
    for section, values in settings["ct_palette"].items():
        base_ct_colors.update(values)
        
    base_ct_colors = {k.replace(" ", "_").replace("/", "_"): v for k, v in base_ct_colors.items()}
    base_ct_colors = {k: v for k, v in base_ct_colors.items() if k in anno_df[groupby_key].unique()}

    h = ma.Heatmap(
        heatmap_df,
        cmap="RdPu",
        vmin=0, vmax=1,
        width=3, height=5,
        rasterized=True,
    )

    h.add_bottom(mp.Labels(heatmap_df.columns, align="center"), pad=0.1)

    h.add_top(mp.Colors(anno_df["condition"], palette={"Control": "lightgrey", tf_query: "#555555"}),
            size=.1, pad=.1)
    h.add_top(mp.Colors(anno_df["annotation"], palette=base_ct_colors),
            size=.1, pad=.01)
    h.add_legends()
    h.render()

    outdir = Path(sc.settings.figdir) / "DE_11122025" / "per_condition_heatmaps"
    outdir.mkdir(parents=True, exist_ok=True)

    plt.savefig(outdir / f"tf_ko_panel_DE_heatmap_per_condition_{tf_query}_down.pdf", bbox_inches="tight")